In [1]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [2]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.176:38087,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
clim_rh = xr.open_dataset(path+'ERA5_clim_RH.nc')
clim_rh

<xarray.Dataset> Size: 478kB
Dimensions:                        (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month                          (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude                       (latitude) float64 656B 7.75 8.0 ... 28.0
  * longitude                      (longitude) float64 968B -89.0 ... -59.0
Data variables:
    __xarray_dataarray_variable__  (month, latitude, longitude) float32 476kB ...

In [4]:
clim_t = xr.open_dataset(path+'ERA5_clim_T.nc')
clim_t

<xarray.Dataset> Size: 478kB
Dimensions:    (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2T     (month, latitude, longitude) float32 476kB ...

In [5]:
tds = xr.open_zarr(path+'sfc_hourly_temp')
tds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2T     (time, latitude, longitude) float32 30GB dask.array<chunksize=(22888, 82, 121), meta=np.ndarray>

In [6]:
rhds = xr.open_dataset(path+'ERA5_RH.nc', chunks={'time': 33})
rhds

<xarray.Dataset> Size: 30GB
Dimensions:                        (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time                           (time) datetime64[ns] 6MB 1940-01-01 ... 2...
  * latitude                       (latitude) float64 656B 7.75 8.0 ... 28.0
  * longitude                      (longitude) float64 968B -89.0 ... -59.0
Data variables:
    __xarray_dataarray_variable__  (time, latitude, longitude) float32 30GB dask.array<chunksize=(33, 82, 121), meta=np.ndarray>

In [13]:
t_anom = tds['VAR_2T'].groupby('time.month') - clim_t
t_anom = t_anom.chunk({'time': 34332})
t_anom

<xarray.Dataset> Size: 30GB
Dimensions:    (latitude: 82, longitude: 121, time: 755304)
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
    month      (time) int64 6MB dask.array<chunksize=(34332,), meta=np.ndarray>
Data variables:
    VAR_2T     (time, latitude, longitude) float32 30GB dask.array<chunksize=(34332, 82, 121), meta=np.ndarray>

In [14]:
rh_anom = rhds.groupby('time.month') - clim_rh
rh_anom = rh_anom.chunk({'time': 34332})
rh_anom

<xarray.Dataset> Size: 30GB
Dimensions:                        (latitude: 82, longitude: 121, time: 755304)
Coordinates:
  * latitude                       (latitude) float64 656B 7.75 8.0 ... 28.0
  * longitude                      (longitude) float64 968B -89.0 ... -59.0
  * time                           (time) datetime64[ns] 6MB 1940-01-01 ... 2...
    month                          (time) int64 6MB dask.array<chunksize=(34332,), meta=np.ndarray>
Data variables:
    __xarray_dataarray_variable__  (time, latitude, longitude) float32 30GB dask.array<chunksize=(34332, 82, 121), meta=np.ndarray>

In [15]:
t_anom.to_zarr(path+'ERA5_t_anom')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 205.54 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [16]:
rh_anom.to_zarr(path+'ERA5_rh_anom')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 204.42 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [17]:
client.shutdown()